In [ ]:
import scanpy as sc

In [ ]:
large = sc.read_h5ad("./data/large_merfish/animal4_int_5cells.h5ad")

In [ ]:
large.var_names

Index(['ENSMUSG00000048482', 'ENSMUSG00000019997', 'ENSMUSG00000026830',
       'ENSMUSG00000031130', 'ENSMUSG00000035805', 'ENSMUSG00000004151',
       'ENSMUSG00000034009', 'ENSMUSG00000026360', 'ENSMUSG00000034714',
       'ENSMUSG00000047259',
       ...
       'ENSMUSG00000042258', 'ENSMUSG00000010476', 'ENSMUSG00000076431',
       'ENSMUSG00000050121', 'ENSMUSG00000031284', 'ENSMUSG00000031364',
       'ENSMUSG00000050511', 'ENSMUSG00000037411', 'ENSMUSG00000038642',
       'ENSMUSG00000054667'],
      dtype='object', length=159)

## Prepare scvi

Keep slices 3 and 6 for testing.

In [ ]:
adata1 = sc.read_h5ad("./data/starmap/starmap_slice_1.h5ad")
adata2 = sc.read_h5ad("./data/starmap/starmap_slice_2.h5ad")
adata3 = sc.read_h5ad("./data/starmap/starmap_slice_3.h5ad")
adata4 = sc.read_h5ad("./data/starmap/starmap_slice_4.h5ad")
adata5 = sc.read_h5ad("./data/starmap/starmap_slice_5.h5ad")
adata6 = sc.read_h5ad("./data/starmap/starmap_slice_6.h5ad")
adata7 = sc.read_h5ad("./data/starmap/starmap_slice_7.h5ad")    
adata8 = sc.read_h5ad("./data/starmap/starmap_slice_8.h5ad")
adata9 = sc.read_h5ad("./data/starmap/starmap_slice_9.h5ad")

train_starmap = adata1.concatenate(adata2, adata4, adata5, adata7, adata8, adata9)

/tmp/ipykernel_1519/1911666138.py:11: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  train_starmap = adata1.concatenate(adata2, adata4, adata5, adata7, adata8, adata9)


In [ ]:
train_starmap.obs['donor_id'] = "starmap_donor"
train_starmap.obs['assay'] = "STARMAP"
train_starmap.obs['dataset'] = "small"


In [115]:

# Map STARMAP gene symbols -> Ensembl IDs, then align features to `large.var_names`
import json
import numpy as np
import scipy.sparse as sp
import anndata as ad

mapping_path = "./data/starmap/starmap_symbol_to_ensembl.json"
with open(mapping_path, "r") as f:
    symbol_to_ensembl = json.load(f)

# Remap var_names (keep only genes that map to Ensembl)
old_symbols = train_starmap.var_names.astype(str)
mapped = np.array([symbol_to_ensembl.get(g, None) for g in old_symbols], dtype=object)
keep_mask = mapped != np.array(None, dtype=object)

train_starmap_mapped = train_starmap[:, keep_mask].copy()
train_starmap_mapped.var_names = mapped[keep_mask].astype(str)
train_starmap_mapped.var_names_make_unique()

# Align to `large` genes: pad missing with zeros, and reorder columns to match `large.var_names`
large_genes = large.var_names.astype(str)
starmap_genes = train_starmap_mapped.var_names.astype(str)
print(train_starmap_mapped.var_names)

missing_in_starmap = large_genes.difference(starmap_genes)
print(f"STARMAP genes after mapping: {train_starmap_mapped.n_vars}")
print(f"Genes in large missing from STARMAP: {len(missing_in_starmap)}")

# Create a new matrix with columns exactly in `large.var_names`
gene_to_idx = {g: i for i, g in enumerate(starmap_genes)}
cols = [gene_to_idx.get(g, -1) for g in large_genes]

X = train_starmap_mapped.X
if not sp.issparse(X):
    X = sp.csr_matrix(X)

n_obs = train_starmap_mapped.n_obs
aligned_blocks = []
for j in cols:
    if j == -1:
        aligned_blocks.append(sp.csr_matrix((n_obs, 1), dtype=X.dtype))
    else:
        aligned_blocks.append(X[:, j])

train_starmap_aligned_X = sp.hstack(aligned_blocks, format="csr")

# Build a NEW AnnData so we don't mutate/overwrite `train_starmap`
train_starmap_aligned = ad.AnnData(
    X=train_starmap_aligned_X,
    obs=train_starmap_mapped.obs.copy(),
    var=large.var.loc[large_genes].copy(),
)
train_starmap_aligned.var_names = large_genes

# Sanity check
assert train_starmap_aligned.var_names.equals(large.var_names)


Index(['ENSMUSG00000070570', 'ENSMUSG00000030218', 'ENSMUSG00000070880',
       'ENSMUSG00000037362', 'ENSMUSG00000021708', 'ENSMUSG00000036192',
       'ENSMUSG00000042589', 'ENSMUSG00000087141', 'ENSMUSG00000006800',
       'ENSMUSG00000019997', 'ENSMUSG00000090223', 'ENSMUSG00000063531',
       'ENSMUSG00000029819', 'ENSMUSG00000004366', 'ENSMUSG00000005716',
       'ENSMUSG00000019772', 'ENSMUSG00000003657', 'ENSMUSG00000032532',
       'ENSMUSG00000042453', 'ENSMUSG00000021250', 'ENSMUSG00000038418',
       'ENSMUSG00000030069', 'ENSMUSG00000037868', 'ENSMUSG00000048482',
       'ENSMUSG00000050953', 'ENSMUSG00000038642', 'ENSMUSG00000041607',
       'ENSMUSG00000029648'],
      dtype='object')
STARMAP genes after mapping: 28
Genes in large missing from STARMAP: 131


In [96]:
train_starmap_aligned.write_h5ad("./data/starmap/train_scvi_starmap.h5ad")

In [97]:
train_starmap_aligned.var_names

Index(['ENSMUSG00000048482', 'ENSMUSG00000019997', 'ENSMUSG00000026830',
       'ENSMUSG00000031130', 'ENSMUSG00000035805', 'ENSMUSG00000004151',
       'ENSMUSG00000034009', 'ENSMUSG00000026360', 'ENSMUSG00000034714',
       'ENSMUSG00000047259',
       ...
       'ENSMUSG00000042258', 'ENSMUSG00000010476', 'ENSMUSG00000076431',
       'ENSMUSG00000050121', 'ENSMUSG00000031284', 'ENSMUSG00000031364',
       'ENSMUSG00000050511', 'ENSMUSG00000037411', 'ENSMUSG00000038642',
       'ENSMUSG00000054667'],
      dtype='object', length=159)

## Prepare merfish

Keep slices 2 and 4 for testing

In [116]:
m_adata1 = sc.read_h5ad("./data/merfish/merfish_slice_1.h5ad")
m_adata2 = sc.read_h5ad("./data/merfish/merfish_slice_2.h5ad")
m_adata3 = sc.read_h5ad("./data/merfish/merfish_slice_3.h5ad")
m_adata4 = sc.read_h5ad("./data/merfish/merfish_slice_4.h5ad")
m_adata5 = sc.read_h5ad("./data/merfish/merfish_slice_5.h5ad")

merfish_train = m_adata1.concatenate(m_adata3, m_adata5)


/tmp/ipykernel_1519/2114409975.py:7: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merfish_train = m_adata1.concatenate(m_adata3, m_adata5)


In [117]:
merfish_train.obs['donor_id'] = "merfish_donor"
merfish_train.obs['assay'] = "MERFISH"
merfish_train.obs['dataset'] = "small"

In [118]:
sequential_genes = [
    "Oxt", "Penk", "Sst", "Tac1", "Gal", "Cartpt", "Vgf", "Trh", 
    "Nts", "Scg2", "Gnrh1", "Tac2", "Cck", "Crh", "Ucn3", 
    "Adcyap1", "Nnat", "Sln", "Mbp", "Th", "Fos"
]

In [120]:

# Remove sequential genes, map MERFISH gene symbols -> Ensembl IDs, then align features to `large.var_names`
import json
import numpy as np
import scipy.sparse as sp
import anndata as ad

merfish_train_no_seq = merfish_train[:, ~merfish_train.var_names.isin(sequential_genes)].copy()

mapping_path = "./data/merfish/merfish_symbol_to_ensembl.json"
with open(mapping_path, "r") as f:
    symbol_to_ensembl = json.load(f)

# Remap var_names (keep only genes that map to Ensembl)
old_symbols = merfish_train_no_seq.var_names.astype(str)
mapped = np.array([symbol_to_ensembl.get(g, None) for g in old_symbols], dtype=object)
keep_mask = mapped != np.array(None, dtype=object)

merfish_train_mapped = merfish_train_no_seq[:, keep_mask].copy()
merfish_train_mapped.var_names = mapped[keep_mask].astype(str)
merfish_train_mapped.var_names_make_unique()

my_list = list(merfish_train_mapped.var_names)

# Align to `large` genes: pad missing with zeros, and reorder columns to match `large.var_names`
large_genes = large.var_names.astype(str)
merfish_genes = merfish_train_mapped.var_names.astype(str)

missing_in_merfish = large_genes.difference(merfish_genes)
print(f"MERFISH genes after seq-gene removal + mapping: {merfish_train_mapped.n_vars}")
print(f"Genes in large missing from MERFISH: {len(missing_in_merfish)}")

# Create a new matrix with columns exactly in `large.var_names`
gene_to_idx = {g: i for i, g in enumerate(merfish_genes)}
cols = [gene_to_idx.get(g, -1) for g in large_genes]

X = merfish_train_mapped.X
if not sp.issparse(X):
    X = sp.csr_matrix(X)

n_obs = merfish_train_mapped.n_obs
aligned_blocks = []
for j in cols:
    if j == -1:
        aligned_blocks.append(sp.csr_matrix((n_obs, 1), dtype=X.dtype))
    else:
        aligned_blocks.append(X[:, j])

merfish_train_aligned_X = sp.hstack(aligned_blocks, format="csr")

# Build a NEW AnnData so we don't mutate/overwrite `merfish_train`
merfish_train_aligned = ad.AnnData(
    X=merfish_train_aligned_X,
    obs=merfish_train_mapped.obs.copy(),
    var=large.var.loc[large_genes].copy(),
)
merfish_train_aligned.var_names = large_genes

# Sanity check
assert merfish_train_aligned.var_names.equals(large.var_names)


MERFISH genes after seq-gene removal + mapping: 135
Genes in large missing from MERFISH: 24


In [121]:
my_list

['ENSMUSG00000015405',
 'ENSMUSG00000020178',
 'ENSMUSG00000030088',
 'ENSMUSG00000048218',
 'ENSMUSG00000074968',
 'ENSMUSG00000024411',
 'ENSMUSG00000046532',
 'ENSMUSG00000036198',
 'ENSMUSG00000020123',
 'ENSMUSG00000031390',
 'ENSMUSG00000025372',
 'ENSMUSG00000048482',
 'ENSMUSG00000008999',
 'ENSMUSG00000031130',
 'ENSMUSG00000023964',
 'ENSMUSG00000031654',
 'ENSMUSG00000024647',
 'ENSMUSG00000029193',
 'ENSMUSG00000030898',
 'ENSMUSG00000000184',
 'ENSMUSG00000047139',
 'ENSMUSG00000023067',
 'ENSMUSG00000045328',
 'ENSMUSG00000021919',
 'ENSMUSG00000020953',
 'ENSMUSG00000058897',
 'ENSMUSG00000039714',
 'ENSMUSG00000024008',
 'ENSMUSG00000027230',
 'ENSMUSG00000021680',
 'ENSMUSG00000018634',
 'ENSMUSG00000003476',
 'ENSMUSG00000032482',
 'ENSMUSG00000021508',
 'ENSMUSG00000032274',
 'ENSMUSG00000024987',
 'ENSMUSG00000028195',
 'ENSMUSG00000062393',
 'ENSMUSG00000010476',
 'ENSMUSG00000037868',
 'ENSMUSG00000026830',
 'ENSMUSG00000019768',
 'ENSMUSG00000004151',
 'ENSMUSG00

In [102]:
merfish_train_aligned.write_h5ad("./data/merfish/train_scvi_merfish.h5ad")

In [104]:
merfish_train_aligned.var_names

Index(['ENSMUSG00000048482', 'ENSMUSG00000019997', 'ENSMUSG00000026830',
       'ENSMUSG00000031130', 'ENSMUSG00000035805', 'ENSMUSG00000004151',
       'ENSMUSG00000034009', 'ENSMUSG00000026360', 'ENSMUSG00000034714',
       'ENSMUSG00000047259',
       ...
       'ENSMUSG00000042258', 'ENSMUSG00000010476', 'ENSMUSG00000076431',
       'ENSMUSG00000050121', 'ENSMUSG00000031284', 'ENSMUSG00000031364',
       'ENSMUSG00000050511', 'ENSMUSG00000037411', 'ENSMUSG00000038642',
       'ENSMUSG00000054667'],
      dtype='object', length=159)

In [105]:
# Save per-slice MERFISH after sequential-gene removal + symbol->Ensembl mapping
import json
import numpy as np
import scanpy as sc

mapping_path = "./data/merfish/merfish_symbol_to_ensembl.json"
with open(mapping_path, "r") as f:
    symbol_to_ensembl = json.load(f)

for i in range(1, 6):
    adata = sc.read_h5ad(f"./data/merfish/merfish_slice_{i}.h5ad")

    # Remove sequential genes (symbols)
    adata = adata[:, ~adata.var_names.isin(sequential_genes)].copy()

    # Map var_names (symbols) -> Ensembl IDs, drop genes that don't map
    old_symbols = adata.var_names.astype(str)
    mapped = np.array([symbol_to_ensembl.get(g, None) for g in old_symbols], dtype=object)
    keep_mask = mapped != np.array(None, dtype=object)

    adata = adata[:, keep_mask].copy()
    adata.var_names = mapped[keep_mask].astype(str)
    adata.var_names_make_unique()

    out_path = f"./data/merfish/merfish_slice_{i}_135genes_ensmbl.h5ad"
    adata.write_h5ad(out_path)
    print(f"Wrote {out_path} with n_vars={adata.n_vars}")


Wrote ./data/merfish/merfish_slice_1_135genes_ensmbl.h5ad with n_vars=135
Wrote ./data/merfish/merfish_slice_2_135genes_ensmbl.h5ad with n_vars=135
Wrote ./data/merfish/merfish_slice_3_135genes_ensmbl.h5ad with n_vars=135
Wrote ./data/merfish/merfish_slice_4_135genes_ensmbl.h5ad with n_vars=135
Wrote ./data/merfish/merfish_slice_5_135genes_ensmbl.h5ad with n_vars=135


## Prepare merfish int data


In [106]:
m_adata1 = sc.read_h5ad("./data/merfish/merfish_slice_1_int.h5ad")
m_adata2 = sc.read_h5ad("./data/merfish/merfish_slice_2_int.h5ad")
m_adata3 = sc.read_h5ad("./data/merfish/merfish_slice_3_int.h5ad")
m_adata4 = sc.read_h5ad("./data/merfish/merfish_slice_4_int.h5ad")
m_adata5 = sc.read_h5ad("./data/merfish/merfish_slice_5_int.h5ad")

merfish_train = m_adata1.concatenate(m_adata3, m_adata5)


/tmp/ipykernel_1519/1862244516.py:7: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merfish_train = m_adata1.concatenate(m_adata3, m_adata5)


In [107]:
merfish_train.obs['donor_id'] = "merfish_donor"
merfish_train.obs['assay'] = "MERFISH"
merfish_train.obs['dataset'] = "small"

In [108]:

# Remove sequential genes, map MERFISH gene symbols -> Ensembl IDs, then align features to `large.var_names`
import json
import numpy as np
import scipy.sparse as sp
import anndata as ad

merfish_train_no_seq = merfish_train[:, ~merfish_train.var_names.isin(sequential_genes)].copy()

mapping_path = "./data/merfish/merfish_symbol_to_ensembl.json"
with open(mapping_path, "r") as f:
    symbol_to_ensembl = json.load(f)

# Remap var_names (keep only genes that map to Ensembl)
old_symbols = merfish_train_no_seq.var_names.astype(str)
mapped = np.array([symbol_to_ensembl.get(g, None) for g in old_symbols], dtype=object)
keep_mask = mapped != np.array(None, dtype=object)

merfish_train_mapped = merfish_train_no_seq[:, keep_mask].copy()
merfish_train_mapped.var_names = mapped[keep_mask].astype(str)
merfish_train_mapped.var_names_make_unique()

# Align to `large` genes: pad missing with zeros, and reorder columns to match `large.var_names`
large_genes = large.var_names.astype(str)
merfish_genes = merfish_train_mapped.var_names.astype(str)

missing_in_merfish = large_genes.difference(merfish_genes)
print(f"MERFISH genes after seq-gene removal + mapping: {merfish_train_mapped.n_vars}")
print(f"Genes in large missing from MERFISH: {len(missing_in_merfish)}")

# Create a new matrix with columns exactly in `large.var_names`
gene_to_idx = {g: i for i, g in enumerate(merfish_genes)}
cols = [gene_to_idx.get(g, -1) for g in large_genes]

X = merfish_train_mapped.X
if not sp.issparse(X):
    X = sp.csr_matrix(X)

n_obs = merfish_train_mapped.n_obs
aligned_blocks = []
for j in cols:
    if j == -1:
        aligned_blocks.append(sp.csr_matrix((n_obs, 1), dtype=X.dtype))
    else:
        aligned_blocks.append(X[:, j])

merfish_train_aligned_X = sp.hstack(aligned_blocks, format="csr")

# Build a NEW AnnData so we don't mutate/overwrite `merfish_train`
merfish_train_aligned = ad.AnnData(
    X=merfish_train_aligned_X,
    obs=merfish_train_mapped.obs.copy(),
    var=large.var.loc[large_genes].copy(),
)
merfish_train_aligned.var_names = large_genes

# Sanity check
assert merfish_train_aligned.var_names.equals(large.var_names)


MERFISH genes after seq-gene removal + mapping: 135
Genes in large missing from MERFISH: 24


In [110]:
merfish_train_aligned.write_h5ad("./data/merfish/train_scvi_merfish_int.h5ad")

In [111]:
merfish_train_aligned.var_names

Index(['ENSMUSG00000048482', 'ENSMUSG00000019997', 'ENSMUSG00000026830',
       'ENSMUSG00000031130', 'ENSMUSG00000035805', 'ENSMUSG00000004151',
       'ENSMUSG00000034009', 'ENSMUSG00000026360', 'ENSMUSG00000034714',
       'ENSMUSG00000047259',
       ...
       'ENSMUSG00000042258', 'ENSMUSG00000010476', 'ENSMUSG00000076431',
       'ENSMUSG00000050121', 'ENSMUSG00000031284', 'ENSMUSG00000031364',
       'ENSMUSG00000050511', 'ENSMUSG00000037411', 'ENSMUSG00000038642',
       'ENSMUSG00000054667'],
      dtype='object', length=159)

In [112]:
# Save per-slice MERFISH after sequential-gene removal + symbol->Ensembl mapping
import json
import numpy as np
import scanpy as sc

mapping_path = "./data/merfish/merfish_symbol_to_ensembl.json"
with open(mapping_path, "r") as f:
    symbol_to_ensembl = json.load(f)

for i in range(1, 6):
    adata = sc.read_h5ad(f"./data/merfish/merfish_slice_{i}_int.h5ad")

    # Remove sequential genes (symbols)
    adata = adata[:, ~adata.var_names.isin(sequential_genes)].copy()

    # Map var_names (symbols) -> Ensembl IDs, drop genes that don't map
    old_symbols = adata.var_names.astype(str)
    mapped = np.array([symbol_to_ensembl.get(g, None) for g in old_symbols], dtype=object)
    keep_mask = mapped != np.array(None, dtype=object)

    adata = adata[:, keep_mask].copy()
    adata.var_names = mapped[keep_mask].astype(str)
    adata.var_names_make_unique()

    out_path = f"./data/merfish/merfish_slice_{i}_135genes_ensmbl_int.h5ad"
    adata.write_h5ad(out_path)
    print(f"Wrote {out_path} with n_vars={adata.n_vars}")


Wrote ./data/merfish/merfish_slice_1_135genes_ensmbl_int.h5ad with n_vars=135
Wrote ./data/merfish/merfish_slice_2_135genes_ensmbl_int.h5ad with n_vars=135
Wrote ./data/merfish/merfish_slice_3_135genes_ensmbl_int.h5ad with n_vars=135
Wrote ./data/merfish/merfish_slice_4_135genes_ensmbl_int.h5ad with n_vars=135
Wrote ./data/merfish/merfish_slice_5_135genes_ensmbl_int.h5ad with n_vars=135
